# Project 33 — LangGraph Incident Summary Flow
## Logs → Severity Classification → Summary → Next Steps

**Stack:** LangGraph · Ollama · Jupyter

In [ ]:
# !pip install -q langgraph langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

class IncidentState(TypedDict):
    raw_logs: str
    severity: str
    affected_services: str
    root_cause_hypothesis: str
    summary: str
    next_steps: str

## Step 2 — Define Incident Nodes

In [ ]:
def classify_severity(state: IncidentState) -> IncidentState:
    class SeverityClassification(BaseModel):
        severity: str = Field(description="SEV1, SEV2, or SEV3")
        affected_services: str
        user_impact: str

    classifier = llm.with_structured_output(SeverityClassification)
    result = classifier.invoke(
        f"Classify this incident from logs:\n\n{state['raw_logs']}"
    )
    print(f"  🚨 Severity: {result.severity}")
    return {"severity": result.severity, "affected_services": result.affected_services}

def analyze_root_cause(state: IncidentState) -> IncidentState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an SRE expert. Analyze these logs and propose the "
         "most likely root cause. Consider: infrastructure, code, configuration, "
         "and external dependencies."),
        ("human", "Logs: {logs}\nAffected: {services}")
    ])
    chain = prompt | llm | StrOutputParser()
    hypothesis = chain.invoke({"logs": state["raw_logs"], "services": state["affected_services"]})
    print(f"  🔍 Root cause analyzed")
    return {"root_cause_hypothesis": hypothesis}

def generate_summary(state: IncidentState) -> IncidentState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Generate a concise incident summary with:
- What happened (1-2 sentences)
- Impact (users, services, duration)
- Root cause hypothesis
- Current status"""),
        ("human", """Severity: {severity}
Affected: {affected}
Root Cause: {root_cause}
Logs: {logs}""")
    ])
    chain = prompt | llm | StrOutputParser()
    summary = chain.invoke({
        "severity": state["severity"], "affected": state["affected_services"],
        "root_cause": state["root_cause_hypothesis"], "logs": state["raw_logs"],
    })
    print(f"  📝 Summary generated")
    return {"summary": summary}

def recommend_next_steps(state: IncidentState) -> IncidentState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Based on the incident, recommend immediate next steps and "
         "longer-term prevention measures. Be specific and actionable."),
        ("human", "Summary: {summary}\nRoot Cause: {root_cause}")
    ])
    chain = prompt | llm | StrOutputParser()
    steps = chain.invoke({"summary": state["summary"], "root_cause": state["root_cause_hypothesis"]})
    print(f"  📋 Next steps recommended")
    return {"next_steps": steps}

## Step 3 — Build Incident Graph

In [ ]:
graph = StateGraph(IncidentState)
graph.add_node("classify", classify_severity)
graph.add_node("root_cause", analyze_root_cause)
graph.add_node("summarize", generate_summary)
graph.add_node("next_steps", recommend_next_steps)

graph.set_entry_point("classify")
graph.add_edge("classify", "root_cause")
graph.add_edge("root_cause", "summarize")
graph.add_edge("summarize", "next_steps")
graph.add_edge("next_steps", END)

app = graph.compile()
print("Incident response workflow compiled!")

## Step 4 — Process Sample Incidents

In [ ]:
incidents = [
    """[2025-01-15 14:23:01] ERROR api-gateway: Connection refused to payment-service:8080
[2025-01-15 14:23:02] ERROR payment-service: OutOfMemoryError: Java heap space
[2025-01-15 14:23:03] ERROR api-gateway: 503 Service Unavailable - /api/checkout
[2025-01-15 14:23:05] WARN load-balancer: payment-service health check failed (3/3)
[2025-01-15 14:23:10] ERROR api-gateway: 1,247 requests failed in last 60 seconds""",

    """[2025-01-15 09:00:01] WARN cron-service: Daily report job started
[2025-01-15 09:15:22] ERROR cron-service: Report generation timeout after 900s
[2025-01-15 09:15:23] WARN cron-service: Retrying report generation (attempt 2/3)
[2025-01-15 09:30:45] ERROR cron-service: Report generation failed — database query timeout
[2025-01-15 09:30:46] INFO notification: Admin notified of report failure""",
]

for i, logs in enumerate(incidents):
    print(f"\n{'='*60}")
    print(f"INCIDENT {i+1}")
    print("="*60)
    result = app.invoke({
        "raw_logs": logs, "severity": "", "affected_services": "",
        "root_cause_hypothesis": "", "summary": "", "next_steps": "",
    })
    print(f"\n--- INCIDENT REPORT ---")
    print(f"Severity: {result['severity']}")
    print(f"\nSummary:\n{result['summary']}")
    print(f"\nNext Steps:\n{result['next_steps']}")

## What You Learned
- **Log-based incident classification** with structured output
- **Root cause analysis** workflow
- **Automated incident reports** with actionable next steps